# Wikidata Entity Linking & Deduplication

Resolve duplicate concepts ("FDR" / "franklin d. roosevelt" / "franklin roosevelt") by linking to canonical Wikidata entities.

Approach:
1. Query Wikidata `wbsearchentities` API for each concept
2. Get the canonical QID (e.g., Q8007 = Franklin D. Roosevelt)
3. Group concepts that map to the same QID
4. Merge duplicates, keeping the most frequent form as canonical
5. Re-score the deduplicated concept list

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import json
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (12, 6)

In [ ]:
# Load the v2 concept table
cs = pd.read_csv("../data/jeopardy_concepts_v2.csv", index_col=0)
print(f"Loaded {len(cs):,} concepts")
print(f"Entity type distribution (freq >= 3):")
print(cs[cs["frequency"] >= 3]["entity_type"].value_counts().to_string())

## 1. Select Candidates for Entity Linking

Focus on named entity types (PERSON, GPE, ORG, WORK_OF_ART, EVENT, LOC, NORP) with frequency >= 3. These are the concepts most likely to have Wikidata entries and most likely to have duplicate variants.

In [ ]:
LINKABLE_TYPES = {"PERSON", "GPE", "ORG", "WORK_OF_ART", "EVENT", "LOC", "NORP", "FAC", "PRODUCT"}

candidates = cs[
    (cs["entity_type"].isin(LINKABLE_TYPES)) & 
    (cs["frequency"] >= 3)
].copy()

print(f"Candidates for Wikidata linking: {len(candidates):,}")
print(f"\nBy type:")
print(candidates["entity_type"].value_counts().to_string())

## 2. Query Wikidata API

Use `wbsearchentities` to find the best matching Wikidata item for each concept. Use the display_name (title case) for better matching.

In [ ]:
WIKIDATA_API = "https://www.wikidata.org/w/api.php"
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "CulturalLiteracyProject/1.0 (research; contact: abrunk@github)"})

def search_wikidata(query, language="en", limit=3):
    """Search Wikidata for an entity. Returns list of {id, label, description}."""
    params = {
        "action": "wbsearchentities",
        "search": query,
        "language": language,
        "limit": limit,
        "format": "json",
    }
    try:
        resp = SESSION.get(WIKIDATA_API, params=params, timeout=10)
        resp.raise_for_status()
        data = resp.json()
        return [
            {
                "qid": r["id"],
                "label": r.get("label", ""),
                "description": r.get("description", ""),
            }
            for r in data.get("search", [])
        ]
    except Exception as e:
        return []

# Quick test
test = search_wikidata("FDR")
print("Test search 'FDR':")
for r in test:
    print(f"  {r['qid']:10s} {r['label']:30s} {r['description']}")

print()
test2 = search_wikidata("Franklin Roosevelt")
print("Test search 'Franklin Roosevelt':")
for r in test2:
    print(f"  {r['qid']:10s} {r['label']:30s} {r['description']}")

In [ ]:
# Helper: pick the best Wikidata result based on entity type
TYPE_DESCRIPTION_HINTS = {
    "PERSON": ["born", "politician", "president", "author", "writer", "actor", "actress",
               "singer", "composer", "scientist", "explorer", "philosopher", "painter",
               "general", "emperor", "king", "queen", "pope", "saint", "leader",
               "american", "british", "french", "german", "roman", "greek",
               "director", "musician", "artist", "inventor", "playwright"],
    "GPE": ["country", "city", "state", "capital", "province", "territory",
            "municipality", "commune", "county", "district", "region",
            "united states", "island nation"],
    "ORG": ["company", "organization", "university", "college", "corporation",
            "brand", "team", "agency", "institution", "foundation", "church"],
    "WORK_OF_ART": ["film", "novel", "play", "book", "song", "opera", "painting",
                     "musical", "album", "poem", "symphony", "series", "show",
                     "television", "video game", "comic"],
    "EVENT": ["war", "battle", "revolution", "conflict", "treaty", "campaign"],
    "LOC": ["river", "mountain", "lake", "ocean", "sea", "island", "continent",
            "desert", "volcano", "canyon", "peninsula", "planet", "body of water"],
    "NORP": ["religion", "ethnic", "people", "mythology", "god", "goddess",
             "deity", "language", "culture", "group"],
}

def pick_best_result(results, entity_type):
    """Pick the best Wikidata result given the expected entity type."""
    if not results:
        return None
    
    hints = TYPE_DESCRIPTION_HINTS.get(entity_type, [])
    
    # Score each result by how many type hints match its description
    scored = []
    for r in results:
        desc = r["description"].lower()
        score = sum(1 for h in hints if h in desc)
        scored.append((score, r))
    
    # If any result has a positive score, use the best one
    scored.sort(key=lambda x: -x[0])
    if scored[0][0] > 0:
        return scored[0][1]
    
    # Otherwise, return the first result (Wikidata's best guess)
    return results[0]

print("Ready.")

In [ ]:
# Load cached linking results from previous API run (avoids ~54 min re-query)
import os

CACHE_PATH = "../data/wikidata_linking.csv"

if os.path.exists(CACHE_PATH):
    print("Loading cached Wikidata linking results...")
    linking_cache = pd.read_csv(CACHE_PATH)
    concept_to_qid = dict(zip(linking_cache["concept"], linking_cache["qid"]))
    concept_to_wiki_label = dict(zip(linking_cache["concept"], linking_cache["wiki_label"]))
    concept_to_wiki_desc = dict(zip(linking_cache["concept"], linking_cache["wiki_description"]))
    failed = [c for c in candidates["concept"] if c not in concept_to_qid]
    print(f"Loaded {len(concept_to_qid):,} cached links")
    print(f"Not linked: {len(failed):,}")
else:
    # Full API run — only needed on first execution
    concept_to_qid = {}
    concept_to_wiki_label = {}
    concept_to_wiki_desc = {}
    failed = []

    display_names = dict(zip(candidates["concept"], candidates["display_name"]))
    entity_types = dict(zip(candidates["concept"], candidates["entity_type"]))
    concept_list = candidates["concept"].tolist()

    total = len(concept_list)
    start_time = time.time()

    for i, concept in enumerate(concept_list):
        search_term = display_names.get(concept, concept)
        etype = entity_types.get(concept, "UNKNOWN")

        results = search_wikidata(search_term)
        best = pick_best_result(results, etype)

        if best:
            concept_to_qid[concept] = best["qid"]
            concept_to_wiki_label[concept] = best["label"]
            concept_to_wiki_desc[concept] = best["description"]
        else:
            failed.append(concept)

        if (i + 1) % 10 == 0:
            time.sleep(1.0)

        if (i + 1) % 500 == 0:
            elapsed = time.time() - start_time
            rate = (i + 1) / elapsed
            eta = (total - i - 1) / rate
            print(f"  {i+1:,} / {total:,} ({100*(i+1)/total:.1f}%) — {rate:.1f}/sec — ETA: {eta/60:.1f} min")

    elapsed = time.time() - start_time
    print(f"\nDone in {elapsed/60:.1f} minutes")
    print(f"Linked: {len(concept_to_qid):,} / {total:,} ({100*len(concept_to_qid)/total:.1f}%)")
    print(f"Failed: {len(failed):,}")

In [ ]:
# Save the raw linking results in case we need to resume
linking_df = pd.DataFrame({
    "concept": list(concept_to_qid.keys()),
    "qid": list(concept_to_qid.values()),
    "wiki_label": [concept_to_wiki_label[c] for c in concept_to_qid],
    "wiki_description": [concept_to_wiki_desc[c] for c in concept_to_qid],
})
linking_df.to_csv("../data/wikidata_linking.csv", index=False)
print(f"Saved {len(linking_df):,} linking results")

## 3. Find Duplicates via QID

Concepts that map to the same Wikidata QID are the same entity. Group them and merge.

In [ ]:
# Find QIDs that have multiple concepts mapping to them
qid_to_concepts = defaultdict(list)
for concept, qid in concept_to_qid.items():
    qid_to_concepts[qid].append(concept)

duplicates = {qid: concepts for qid, concepts in qid_to_concepts.items() if len(concepts) > 1}

print(f"QIDs with multiple concept forms: {len(duplicates):,}")
print(f"Total concepts involved: {sum(len(c) for c in duplicates.values()):,}")

# Get frequency for each concept
freq_lookup = dict(zip(cs["concept"], cs["frequency"]))

# Show the duplicates, sorted by combined frequency
dup_list = []
for qid, concepts in duplicates.items():
    total_freq = sum(freq_lookup.get(c, 0) for c in concepts)
    canonical = max(concepts, key=lambda c: freq_lookup.get(c, 0))
    variants = [c for c in concepts if c != canonical]
    wiki_label = concept_to_wiki_label.get(canonical, "")
    dup_list.append({
        "qid": qid,
        "wiki_label": wiki_label,
        "canonical": canonical,
        "variants": variants,
        "total_freq": total_freq,
        "n_forms": len(concepts),
    })

dup_list.sort(key=lambda x: -x["total_freq"])

print(f"\nTop 40 duplicates by combined frequency:")
for d in dup_list[:40]:
    variant_str = ", ".join(f"{v} ({freq_lookup.get(v,0)})" for v in d["variants"])
    print(f"  {d['qid']:10s} {d['canonical']:30s} ({freq_lookup.get(d['canonical'],0):>3}) <- {variant_str}")

In [ ]:
# Build the merge map: variant -> canonical (most frequent form)
wiki_alias_map = {}
for qid, concepts in duplicates.items():
    canonical = max(concepts, key=lambda c: freq_lookup.get(c, 0))
    for c in concepts:
        if c != canonical:
            wiki_alias_map[c] = canonical

print(f"Wikidata-based alias map: {len(wiki_alias_map):,} merges")

# Show some interesting merges
print("\nSample merges:")
shown = 0
for variant, canonical in sorted(wiki_alias_map.items(), key=lambda x: -(freq_lookup.get(x[0],0) + freq_lookup.get(x[1],0))):
    if shown >= 30:
        break
    print(f"  {variant:30s} ({freq_lookup.get(variant,0):>3}) -> {canonical:30s} ({freq_lookup.get(canonical,0):>3})")
    shown += 1

In [ ]:
# Manual review blocklist: merges where Wikidata matched the wrong entity
# (ambiguous short strings, different places/people incorrectly linked, etc.)
BLOCKED_MERGES = {
    "mexico city",           # different place than mexico
    "federal",               # unrelated to germany
    "che",                   # unrelated to switzerland
    "18th",                  # unrelated to louisiana
    "lincoln, nebraska",     # city vs the broader "lincoln" concept
    "pan",                   # Greek god, not panama
    "quebec city",           # different place than quebec (province)
    "edo",                   # historical name but distinct Jeopardy answer from tokyo
    "mc",                    # unrelated to monaco
    "iq",                    # unrelated to iraq
    "king charles ii",       # different person than prince charles
    "charlie",               # too ambiguous for charlie chaplin
    "guy",                   # unrelated to guyana
    "joe cool",              # Snoopy alter ego, not joe montana
    "kingston jamaica",      # wrong entity (mapped to "kings")
    "meta",                  # distinct from facebook as Jeopardy answer
    "gall",                  # unrelated to chickens
    "choir",                 # different word than chorus, not a dedup
    "fa",                    # unrelated to farsi
    "das boot",              # film, not the same as u-boat
    "easter seals",          # different org than american lung association
    "texas ranger",          # wrong entity (mapped to washington senators)
    "judy jetson",           # different character than george jetson
}

for variant in BLOCKED_MERGES:
    wiki_alias_map.pop(variant, None)

print(f"Blocked {len(BLOCKED_MERGES)} bad merges")
print(f"Remaining alias map: {len(wiki_alias_map):,} merges")

## 4. Apply Merges and Rebuild

Apply the Wikidata dedup on top of the previous alias map, rebuild stats, and re-score.

In [ ]:
# Load raw data again to rebuild from scratch with all dedup layers
import re

df = pd.read_csv(
    "../data/jeopardy_clue_dataset/combined_season1-41.tsv",
    sep="\t",
    dtype={"clue_value": "Int64", "daily_double_value": "Int64"},
    parse_dates=["air_date"],
    low_memory=False,
)
df["season"] = df["air_date"].apply(lambda d: d.year if d.month >= 9 else d.year - 1)
df = df.dropna(subset=["question"])

def normalize_response(text):
    if not isinstance(text, str):
        return ""
    s = text.strip()
    s = re.sub(r"\(([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)\)", r"\1", s)
    s = re.sub(r"\s*\(or\s+[^)]+\)\s*$", "", s)
    s = s.strip('"\'')
    s = s.lower().strip()
    s = re.sub(r"^(what|who|where|when)\s+(is|are|was|were|does)\s+", "", s)
    s = re.sub(r"^(the|a|an)\s+", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["concept"] = df["question"].apply(normalize_response)

# Load the previous alias map from notebook 03
# (We stored it implicitly — let's reload the v2 data to get the mapping)
# For simplicity, apply the Wikidata alias map on top of normalized concepts
df["concept_final"] = df["concept"].map(lambda c: wiki_alias_map.get(c, c))

print(f"Unique concepts before Wikidata dedup: {df['concept'].nunique():,}")
print(f"Unique concepts after Wikidata dedup: {df['concept_final'].nunique():,}")
print(f"Merged: {df['concept'].nunique() - df['concept_final'].nunique():,}")

In [ ]:
# Rebuild concept stats
concept_stats = df.groupby("concept_final").agg(
    frequency=("concept_final", "size"),
    seasons_appeared=("season", "nunique"),
    category_breadth=("category", "nunique"),
    avg_clue_value=("clue_value", "mean"),
    first_seen=("air_date", "min"),
    last_seen=("air_date", "max"),
).reset_index()
concept_stats.rename(columns={"concept_final": "concept"}, inplace=True)

# Carry forward entity types and display names from v2
v2_types = dict(zip(cs["concept"], cs["entity_type"]))
v2_display = dict(zip(cs["concept"], cs["display_name"]))
concept_stats["entity_type"] = concept_stats["concept"].map(v2_types).fillna("UNKNOWN")
concept_stats["display_name"] = concept_stats["concept"].map(v2_display)

# Add Wikidata QID and canonical label where available
concept_stats["wikidata_qid"] = concept_stats["concept"].map(concept_to_qid)
concept_stats["wiki_label"] = concept_stats["concept"].map(concept_to_wiki_label)
concept_stats["wiki_description"] = concept_stats["concept"].map(concept_to_wiki_desc)

# Re-score
def min_max_normalize(series):
    rng = series.max() - series.min()
    if rng == 0:
        return series * 0
    return (series - series.min()) / rng

concept_stats["freq_score"] = min_max_normalize(np.log1p(concept_stats["frequency"]))
concept_stats["season_score"] = min_max_normalize(concept_stats["seasons_appeared"])
concept_stats["breadth_score"] = min_max_normalize(np.log1p(concept_stats["category_breadth"]))

concept_stats["importance"] = (
    0.40 * concept_stats["freq_score"] +
    0.35 * concept_stats["season_score"] +
    0.25 * concept_stats["breadth_score"]
)

# Within-type scores
concept_stats["importance_within_type"] = 0.0
for etype in concept_stats["entity_type"].unique():
    mask = concept_stats["entity_type"] == etype
    subset = concept_stats.loc[mask].copy()
    if len(subset) < 2:
        continue
    subset["fs"] = min_max_normalize(np.log1p(subset["frequency"]))
    subset["ss"] = min_max_normalize(subset["seasons_appeared"])
    subset["bs"] = min_max_normalize(np.log1p(subset["category_breadth"]))
    subset["imp"] = 0.40 * subset["fs"] + 0.35 * subset["ss"] + 0.25 * subset["bs"]
    concept_stats.loc[mask, "importance_within_type"] = subset["imp"].values

concept_stats = concept_stats.sort_values("importance", ascending=False).reset_index(drop=True)
concept_stats.index += 1
concept_stats.index.name = "rank"

print(f"Final concept table: {len(concept_stats):,} concepts")
print(f"\nTop 30:")
print(concept_stats.head(30)[["concept", "display_name", "entity_type", "frequency", "seasons_appeared", "wikidata_qid", "importance"]].to_string())

In [ ]:
# Show top 15 PERSONs after Wikidata dedup — did the merges help?
print("Top 20 PERSONs after Wikidata dedup:")
persons = concept_stats[concept_stats["entity_type"] == "PERSON"].head(20)
print(persons[["concept", "display_name", "frequency", "seasons_appeared", "wikidata_qid", "wiki_label", "importance"]].to_string())

## 5. Validate Merges

Check for any bad merges — cases where two different concepts were incorrectly linked to the same QID.

In [ ]:
# Spot-check: concepts that gained the most frequency from merging
v2_freq = dict(zip(cs["concept"], cs["frequency"]))
concept_stats["freq_gain"] = concept_stats.apply(
    lambda r: r["frequency"] - v2_freq.get(r["concept"], r["frequency"]), axis=1
)

gained = concept_stats[concept_stats["freq_gain"] > 0].sort_values("freq_gain", ascending=False)
print(f"Concepts that gained frequency from Wikidata merging: {len(gained):,}")
print(f"\nTop 30 by frequency gain:")
print(gained.head(30)[["concept", "display_name", "entity_type", "frequency", "freq_gain", "wikidata_qid", "wiki_label"]].to_string())

## 6. Export Final Data

In [ ]:
# Save final concept table
output_path = "../data/jeopardy_concepts_v3.csv"
export_cols = [
    "concept", "display_name", "entity_type", "frequency", "seasons_appeared",
    "category_breadth", "avg_clue_value", "first_seen", "last_seen",
    "importance", "importance_within_type",
    "wikidata_qid", "wiki_label", "wiki_description",
]
concept_stats[export_cols].to_csv(output_path, index=True)
print(f"Saved {len(concept_stats):,} concepts to {output_path}")

# Summary
print(f"\n{'='*60}")
print(f"FINAL SUMMARY (v3 — with Wikidata dedup)")
print(f"{'='*60}")
print(f"Total concepts: {len(concept_stats):,}")
print(f"With Wikidata QID: {concept_stats['wikidata_qid'].notna().sum():,}")
print(f"Entity type coverage: {100*(1 - (concept_stats['entity_type']=='UNKNOWN').mean()):.1f}%")
print(f"Concepts appearing 5+ times: {(concept_stats['frequency'] >= 5).sum():,}")
print(f"Concepts in 20+ seasons: {(concept_stats['seasons_appeared'] >= 20).sum():,}")
print(f"\nWikidata merges applied: {len(wiki_alias_map):,}")
print(f"\nType breakdown (freq >= 5):")
print(concept_stats[concept_stats["frequency"] >= 5]["entity_type"].value_counts().to_string())